In [2]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 11.3 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.0
    Uninstalling pyarrow-19.0.0:
      Successfully uninstalled pyarrow-19.0.0
  Attempting uninstall: dill0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [pyarrow]
    Found existing installation: dill 0.3.8━━━━━━━━━━━━━━━━━━━ 1/5 [pyarrow]
    Uninstalling dill-0.3.8:90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [pyarrow]
      Successfully uninstalled dill-0.3.8━━━━━━━━━━━━━━━━━━━━━ 1/5 [pyarrow]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [datasets]4/5 [datasets]ess]


In [2]:
from datasets import load_dataset
from pathlib import Path
from tqdm.auto import tqdm
import re
import unicodedata
import random


### TinyStories N-gram preprocessing

In [5]:

def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)

    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"')
    text = text.replace("”", '"')

    return text


def split_story_into_sentences(text):
    text = normalize_text(text)
    text = text.replace("\n", " ")

    sentences = re.split(r"[.!?]+", text)

    return sentences


def clean_sentence(sentence):
    sentence = normalize_text(sentence)
    sentence = sentence.lower()

    sentence = sentence.replace("'", "")

    sentence = re.sub(r"[^a-z0-9\s]", " ", sentence)

    sentence = re.sub(r"\s+", " ", sentence).strip()

    return sentence


def story_to_clean_sentences(story_text):
    raw_sentences = split_story_into_sentences(story_text)

    cleaned_sentences = []

    for sentence in raw_sentences:
        cleaned = clean_sentence(sentence)

        if cleaned:
            cleaned_sentences.append(cleaned)

    return cleaned_sentences


def save_lines(lines, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")


def convert_stories_to_sentences(stories):
    all_sentences = []

    for story in stories:
        sentences = story_to_clean_sentences(story)
        all_sentences.extend(sentences)

    return all_sentences



In [ ]:

def main():
    output_dir = Path("data")
    output_dir.mkdir(parents=True, exist_ok=True)

    dataset = load_dataset("roneneldan/TinyStories", split="train[:1000000]")

    stories = []

    for row in tqdm(dataset, desc="Loading TinyStories"):
        text = row["text"].strip()

        if text:
            stories.append(text)

    random.seed(42)
    random.shuffle(stories)

    n = len(stories)

    train_end = int(0.8 * n)
    val_end = int(0.81 * n)

    train_stories = stories[:train_end]
    val_stories = stories[train_end:val_end]
    test_stories = stories[val_end:]

    train_sentences = convert_stories_to_sentences(train_stories)
    val_sentences = convert_stories_to_sentences(val_stories)
    test_sentences = convert_stories_to_sentences(test_stories)

    save_lines(train_sentences, "data/tiny_stories/tinystories_train.txt")
    save_lines(val_sentences, "data/tiny_stories/tinystories_val.txt")
    save_lines(test_sentences, "data/tiny_stories/tinystories_test.txt")
    print("Saved splits:")
    print("Train stories:", len(train_stories))
    print("Val stories:  ", len(val_stories))
    print("Test stories: ", len(test_stories))
    print()
    print("Train sentences:", len(train_sentences))
    print("Val sentences:  ", len(val_sentences))
    print("Test sentences: ", len(test_sentences))


if __name__ == "__main__":
    main()

Loading TinyStories:   0%|          | 0/1000000 [00:00<?, ?it/s]

Saved splits:
Train stories: 799943
Val stories:   9999
Test stories:  189987

Train sentences: 15603516
Val sentences:   193167
Test sentences:  3706169


### WikiText-2 N-gram preprocessing

In [6]:



def split_text_into_sentences(text):
    text = normalize_text(text)
    text = text.replace("\n", " ")

    sentences = re.split(r"[.!?]+", text)

    return sentences




def text_to_clean_sentences(text):
    raw_sentences = split_text_into_sentences(text)

    cleaned_sentences = []

    for sentence in raw_sentences:
        cleaned = clean_sentence(sentence)

        if cleaned:
            cleaned_sentences.append(cleaned)

    return cleaned_sentences




def convert_wikitext_split_to_sentences(dataset_split, split_name):
    all_sentences = []

    for row in tqdm(dataset_split, desc=f"Processing WikiText-2 {split_name}"):
        text = row["text"].strip()

        if not text:
            continue

        if text.startswith("=") and text.endswith("="):
            continue

        sentences = text_to_clean_sentences(text)
        all_sentences.extend(sentences)

    return all_sentences


In [ ]:


def main():
    output_dir = Path("data/wikitext_2")
    output_dir.mkdir(parents=True, exist_ok=True)

    train_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="train",
    )

    val_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="validation",
    )

    test_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="test",
    )

    train_sentences = convert_wikitext_split_to_sentences(
        train_dataset,
        split_name="train",
    )

    val_sentences = convert_wikitext_split_to_sentences(
        val_dataset,
        split_name="validation",
    )

    test_sentences = convert_wikitext_split_to_sentences(
        test_dataset,
        split_name="test",
    )

    save_lines(
        train_sentences,
        output_dir / "wikitext2_train.txt",
    )

    save_lines(
        val_sentences,
        output_dir / "wikitext2_val.txt",
    )

    save_lines(
        test_sentences,
        output_dir / "wikitext2_test.txt",
    )

    print("Saved WikiText-2 splits:")
    print("Train sentences:", len(train_sentences))
    print("Val sentences:  ", len(val_sentences))
    print("Test sentences: ", len(test_sentences))
    print()
    print("Saved to:")
    print(output_dir / "wikitext2_train.txt")
    print(output_dir / "wikitext2_val.txt")
    print(output_dir / "wikitext2_test.txt")


if __name__ == "__main__":
    main()

Processing WikiText-2 train:   0%|          | 0/36718 [00:00<?, ?it/s]

Processing WikiText-2 validation:   0%|          | 0/3760 [00:00<?, ?it/s]

Processing WikiText-2 test:   0%|          | 0/4358 [00:00<?, ?it/s]

Saved WikiText-2 splits:
Train sentences: 85757
Val sentences:   9046
Test sentences:  10434

Saved to:
data/wikitext_2/wikitext2_train.txt
data/wikitext_2/wikitext2_val.txt
data/wikitext_2/wikitext2_test.txt


### TinyStories Transformer preprocessing

In [11]:
def clean_text_for_transformer(text):
    text = normalize_text(text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def convert_stories_to_transformer_texts(stories):
    transformer_texts = []

    for story in stories:
        cleaned = clean_text_for_transformer(story)

        if cleaned:
            transformer_texts.append(cleaned)

    return transformer_texts


def prepare_tinystories_for_transformer():
    output_dir = Path("data/tiny_stories_transformer")
    output_dir.mkdir(parents=True, exist_ok=True)

    dataset = load_dataset("roneneldan/TinyStories", split="train[:1000000]")

    stories = []

    for row in tqdm(dataset, desc="Loading TinyStories for Transformer"):
        text = row["text"].strip()

        if text:
            stories.append(text)

    random.seed(42)
    random.shuffle(stories)

    n = len(stories)

    train_end = int(0.8 * n)
    val_end = int(0.81 * n)

    train_stories = stories[:train_end]
    val_stories = stories[train_end:val_end]
    test_stories = stories[val_end:]

    train_texts = convert_stories_to_transformer_texts(train_stories)
    val_texts = convert_stories_to_transformer_texts(val_stories)
    test_texts = convert_stories_to_transformer_texts(test_stories)

    save_lines(
        train_texts,
        output_dir / "tinystories_transformer_train.txt",
    )
    save_lines(
        val_texts,
        output_dir / "tinystories_transformer_val.txt",
    )
    save_lines(
        test_texts,
        output_dir / "tinystories_transformer_test.txt",
    )

    print("Saved TinyStories Transformer splits:")
    print("Train stories:", len(train_stories))
    print("Val stories:  ", len(val_stories))
    print("Test stories: ", len(test_stories))
    print()
    print("Transformer train texts:", len(train_texts))
    print("Transformer val texts:  ", len(val_texts))
    print("Transformer test texts: ", len(test_texts))
    print()
    print("Saved to:")
    print(output_dir / "tinystories_transformer_train.txt")
    print(output_dir / "tinystories_transformer_val.txt")
    print(output_dir / "tinystories_transformer_test.txt")


prepare_tinystories_for_transformer()

Loading TinyStories for Transformer:   0%|          | 0/1000000 [00:00<?, ?it/s]

Saved TinyStories Transformer splits:
Train stories: 799943
Val stories:   9999
Test stories:  189987

Transformer train texts: 799943
Transformer val texts:   9999
Transformer test texts:  189987

Saved to:
data/tiny_stories_transformer/tinystories_transformer_train.txt
data/tiny_stories_transformer/tinystories_transformer_val.txt
data/tiny_stories_transformer/tinystories_transformer_test.txt


### WikiText-2 Transformer preprocessing

In [12]:

def convert_wikitext_split_to_transformer_texts(dataset_split, split_name):
    transformer_texts = []

    for row in tqdm(dataset_split, desc=f"Processing WikiText-2 Transformer {split_name}"):
        text = row["text"].strip()

        if not text:
            continue

        if text.startswith("=") and text.endswith("="):
            continue

        cleaned = clean_text_for_transformer(text)

        if cleaned:
            transformer_texts.append(cleaned)

    return transformer_texts


def prepare_wikitext2_for_transformer():
    output_dir = Path("data/wikitext_2_transformer")
    output_dir.mkdir(parents=True, exist_ok=True)

    train_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="train",
    )

    val_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="validation",
    )

    test_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="test",
    )

    train_texts = convert_wikitext_split_to_transformer_texts(
        train_dataset,
        split_name="train",
    )

    val_texts = convert_wikitext_split_to_transformer_texts(
        val_dataset,
        split_name="validation",
    )

    test_texts = convert_wikitext_split_to_transformer_texts(
        test_dataset,
        split_name="test",
    )

    save_lines(
        train_texts,
        output_dir / "wikitext2_transformer_train.txt",
    )

    save_lines(
        val_texts,
        output_dir / "wikitext2_transformer_val.txt",
    )

    save_lines(
        test_texts,
        output_dir / "wikitext2_transformer_test.txt",
    )

    print("Saved WikiText-2 Transformer splits:")
    print("Train texts:", len(train_texts))
    print("Val texts:  ", len(val_texts))
    print("Test texts: ", len(test_texts))
    print()
    print("Saved to:")
    print(output_dir / "wikitext2_transformer_train.txt")
    print(output_dir / "wikitext2_transformer_val.txt")
    print(output_dir / "wikitext2_transformer_test.txt")


prepare_wikitext2_for_transformer()

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Processing WikiText-2 Transformer train:   0%|          | 0/36718 [00:00<?, ?it/s]

Processing WikiText-2 Transformer validation:   0%|          | 0/3760 [00:00<?, ?it/s]

Processing WikiText-2 Transformer test:   0%|          | 0/4358 [00:00<?, ?it/s]

Saved WikiText-2 Transformer splits:
Train texts: 17556
Val texts:   1841
Test texts:  2185

Saved to:
data/wikitext_2_transformer/wikitext2_transformer_train.txt
data/wikitext_2_transformer/wikitext2_transformer_val.txt
data/wikitext_2_transformer/wikitext2_transformer_test.txt


# MobileText SMS

## MobileText preprocessing for N-gram

In [1]:
from pathlib import Path
from itertools import islice
from tqdm.auto import tqdm
import re
import unicodedata


MOBILETEXT_TRAIN_FRACTION = 0.4

MOBILETEXT_SOURCE_LINE_COUNTS = {
    "train": {
        "mobile": 1_131_886,
        "non-mobile": 10_753_223,
    },
    "validate": {
        "mobile": 33_804,
        "non-mobile": 344_151,
    },
    "test": {
        "mobile": 23_666,
        "non-mobile": 213_516,
    },
}


MOBILETEXT_SPLITS = {
    "train": {
        "source_split": "train",
        "source_suffix": "train",
        "output_name": "train_sms.txt",
        "expected_lines": 5_942_554,
        "source_fraction": MOBILETEXT_TRAIN_FRACTION,
    },
    "validate": {
        "source_split": "validate",
        "source_suffix": "dev",
        "output_name": "validate_sms.txt",
        "expected_lines": 377_955,
        "source_fraction": 1.0,
    },
    "test": {
        "source_split": "test",
        "source_suffix": "test",
        "output_name": "test_sms.txt",
        "expected_lines": 237_182,
        "source_fraction": 1.0,
    },
}


def get_project_data_dir():
    if Path("data/Data_unclean/mobiletext").exists():
        return Path("data")
    if Path("scr/data/Data_unclean/mobiletext").exists():
        return Path("scr/data")
    return Path("data")


def get_mobiletext_root():
    candidates = [
        Path("data/Data_unclean/mobiletext"),
        Path("scr/data/Data_unclean/mobiletext"),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError("Could not find data/Data_unclean/mobiletext")


def mobiletext_source_paths(source_suffix):
    root = get_mobiletext_root()
    paths = [
        root / "sets" / f"mobile_{source_suffix}.txt",
        root / "sets" / f"non-mobile_{source_suffix}.txt",
    ]

    missing = [path for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing MobileText split files: {missing}")

    return paths


def mobiletext_source_type(source_path):
    if source_path.name.startswith("mobile_"):
        return "mobile"
    if source_path.name.startswith("non-mobile_"):
        return "non-mobile"
    raise ValueError(f"Unexpected MobileText source file: {source_path}")


def mobiletext_source_line_limit(config, source_path):
    source_type = mobiletext_source_type(source_path)
    source_count = MOBILETEXT_SOURCE_LINE_COUNTS[config["source_split"]][source_type]
    return int(source_count * config.get("source_fraction", 1.0))


def normalize_mobile_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"')
    text = text.replace("”", '"')
    return text


def clean_mobile_sentence_for_ngram(text):
    text = normalize_mobile_text(text)
    text = text.lower()
    text = text.replace("'", "")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def write_mobile_ngram_split(split_name, config, output_dir):
    output_path = output_dir / config["output_name"]
    written = 0
    read = 0

    with output_path.open("w", encoding="utf-8") as dst:
        for source_path in mobiletext_source_paths(config["source_suffix"]):
            line_limit = mobiletext_source_line_limit(config, source_path)
            with source_path.open("r", encoding="utf-8") as src:
                for line in tqdm(islice(src, line_limit), desc=f"N-gram {split_name}: {source_path.name}", total=line_limit):
                    read += 1
                    cleaned = clean_mobile_sentence_for_ngram(line.rstrip("\n"))

                    if not cleaned:
                        continue

                    dst.write(cleaned + "\n")
                    written += 1

    return read, written, output_path


def prepare_mobiletext_for_ngram():
    output_dir = get_project_data_dir() / "mobile_ngram"
    output_dir.mkdir(parents=True, exist_ok=True)

    print("MobileText source:", get_mobiletext_root())
    print("N-gram output:", output_dir)
    print()

    for split_name, config in MOBILETEXT_SPLITS.items():
        read, written, output_path = write_mobile_ngram_split(split_name, config, output_dir)
        print(f"{split_name}: read={read:,} expected={config['expected_lines']:,} written={written:,}")
        print("  saved:", output_path)


prepare_mobiletext_for_ngram()

MobileText source: data/Data_unclean/mobiletext
N-gram output: data/mobile_ngram



N-gram train: mobile_train.txt:   0%|          | 0/452754 [00:00<?, ?it/s]

N-gram train: non-mobile_train.txt:   0%|          | 0/4301289 [00:00<?, ?it/s]

train: read=4,754,043 expected=5,942,554 written=4,754,043
  saved: data/mobile_ngram/train_sms.txt


N-gram validate: mobile_dev.txt:   0%|          | 0/33804 [00:00<?, ?it/s]

N-gram validate: non-mobile_dev.txt:   0%|          | 0/344151 [00:00<?, ?it/s]

validate: read=377,955 expected=377,955 written=377,955
  saved: data/mobile_ngram/validate_sms.txt


N-gram test: mobile_test.txt:   0%|          | 0/23666 [00:00<?, ?it/s]

N-gram test: non-mobile_test.txt:   0%|          | 0/213516 [00:00<?, ?it/s]

test: read=237,182 expected=237,182 written=237,182
  saved: data/mobile_ngram/test_sms.txt


## MobileText preprocessing for Transformer

In [2]:
def clean_mobile_sentence_for_transformer(text):
    text = normalize_mobile_text(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def write_mobile_transformer_split(split_name, config, output_dir):
    output_path = output_dir / config["output_name"]
    written = 0
    read = 0

    with output_path.open("w", encoding="utf-8") as dst:
        for source_path in mobiletext_source_paths(config["source_suffix"]):
            line_limit = mobiletext_source_line_limit(config, source_path)
            with source_path.open("r", encoding="utf-8") as src:
                for line in tqdm(islice(src, line_limit), desc=f"Transformer {split_name}: {source_path.name}", total=line_limit):
                    read += 1
                    cleaned = clean_mobile_sentence_for_transformer(line.rstrip("\n"))

                    if not cleaned:
                        continue

                    dst.write(cleaned + "\n")
                    written += 1

    return read, written, output_path


def prepare_mobiletext_for_transformer():
    output_dir = get_project_data_dir() / "mobile_transformers"
    output_dir.mkdir(parents=True, exist_ok=True)

    print("MobileText source:", get_mobiletext_root())
    print("Transformer output:", output_dir)
    print()

    for split_name, config in MOBILETEXT_SPLITS.items():
        read, written, output_path = write_mobile_transformer_split(split_name, config, output_dir)
        print(f"{split_name}: read={read:,} expected={config['expected_lines']:,} written={written:,}")
        print("  saved:", output_path)


prepare_mobiletext_for_transformer()

MobileText source: data/Data_unclean/mobiletext
Transformer output: data/mobile_transformers



Transformer train: mobile_train.txt:   0%|          | 0/452754 [00:00<?, ?it/s]

Transformer train: non-mobile_train.txt:   0%|          | 0/4301289 [00:00<?, ?it/s]

train: read=4,754,043 expected=5,942,554 written=4,754,043
  saved: data/mobile_transformers/train_sms.txt


Transformer validate: mobile_dev.txt:   0%|          | 0/33804 [00:00<?, ?it/s]

Transformer validate: non-mobile_dev.txt:   0%|          | 0/344151 [00:00<?, ?it/s]

validate: read=377,955 expected=377,955 written=377,955
  saved: data/mobile_transformers/validate_sms.txt


Transformer test: mobile_test.txt:   0%|          | 0/23666 [00:00<?, ?it/s]

Transformer test: non-mobile_test.txt:   0%|          | 0/213516 [00:00<?, ?it/s]

test: read=237,182 expected=237,182 written=237,182
  saved: data/mobile_transformers/test_sms.txt


### N-gram training and evaluation

In [ ]:
%%bash
set -e

if [ -d "scr/ngram" ]; then
  PROJECT_ROOT="$(pwd)"
elif [ -d "../scr/ngram" ]; then
  PROJECT_ROOT="$(cd .. && pwd)"
else
  PROJECT_ROOT="/Users/hej/NLP-project"
fi

cd "$PROJECT_ROOT"

if [ -f "scr/data/mobile_ngram/train_sms.txt" ]; then
  NGRAM_DATA_DIR="scr/data/mobile_ngram"
else
  NGRAM_DATA_DIR="data/mobile_ngram"
fi

python scr/ngram/ngram_train.py \
  --train_path "$NGRAM_DATA_DIR/train_sms.txt" \
  --save_path models/ngram/mobile_sms_ngram_model.pkl \
  --max_n_gram 4 \
  --min_count 3

python scr/ngram/ngram_evaluate.py \
  --project_root "$PROJECT_ROOT" \
  --model_path models/ngram/mobile_sms_ngram_model.pkl \
  --val_path "$NGRAM_DATA_DIR/validate_sms.txt" \
  --best_lambdas_path results/metrics/best_ngram_lambdas_mobile_sms.json \
  --validation_results_path results/metrics/ngram_validation_results_mobile_sms.json \
  --best_lambdas_plot_path results/plots/best_ngram_lambdas_mobile_sms.png \
  --top_lambdas_plot_path results/plots/top_ngram_lambdas_mobile_sms.png \
  --max_val_sentences 1000

LAMBDAS=$(python -c 'import json; p="results/metrics/best_ngram_lambdas_mobile_sms.json"; d=json.load(open(p))["best_lambdas"]; print(",".join(str(d[str(i)]) for i in range(1, len(d)+1)))')

python scr/ngram/ngram_test.py \
  --project_root "$PROJECT_ROOT" \
  --model_path models/ngram/mobile_sms_ngram_model.pkl \
  --test_path "$NGRAM_DATA_DIR/test_sms.txt" \
  --test_results_path results/metrics/ngram_test_results_mobile_sms.json \
  --lambdas "$LAMBDAS" \
  --max_test_sentences 3000

Loaded preprocessed training data
Number of sentences: 11885109


Training n-gram model: 100%|█████| 11885109/11885109 [25:04<00:00, 7902.02it/s]
